# Synthetic single-cell clustering

Simulate a 500-cell RNA-seq dataset (3 cell types), then run normalize → PCA → KMeans → t-SNE, check the Adjusted Rand Index and inspect marker genes.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score

rng = np.random.default_rng(7)
n_cells, n_genes, n_types = 500, 150, 3
sizes = [int(n_cells * s) for s in (0.40, 0.35, 0.25)]
sizes[-1] = n_cells - sum(sizes[:-1])
labels = np.repeat(np.arange(n_types), sizes)
rng.shuffle(labels)

base = np.exp(rng.uniform(-5, -0.5, n_genes))
rates = np.repeat(base[None], n_cells, axis=0).copy()
markers = []
for t in range(n_types):
    idx = rng.choice(n_genes, 8, replace=False); markers.append(idx)
    rates[np.ix_(labels == t, idx)] *= rng.uniform(8, 20, size=idx.size)
counts = np.zeros((n_cells, n_genes), dtype=np.int32)
for i in range(n_cells):
    counts[i] = rng.multinomial(int(rng.uniform(800, 4000)), rates[i] / rates[i].sum())

In [ ]:
cpm = counts / counts.sum(axis=1, keepdims=True) * 1e4
norm = np.log1p(cpm)
pc = PCA(n_components=15, random_state=7).fit_transform(norm)
cluster = KMeans(n_clusters=3, n_init=10, random_state=7).fit_predict(pc)
emb = TSNE(n_components=2, perplexity=30, random_state=7).fit_transform(pc)

ari = adjusted_rand_score(labels, cluster)
print(f'Adjusted Rand Index = {ari:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
axes[0].scatter(emb[:, 0], emb[:, 1], c=cluster, cmap='tab10', s=9)
axes[0].set_title('KMeans clusters')
axes[1].scatter(emb[:, 0], emb[:, 1], c=labels, cmap='Set1', s=9)
axes[1].set_title('Ground truth')

In [ ]:
order = np.argsort(norm.std(axis=0))[::-1][:12]
idx = np.linspace(0, n_cells - 1, 80).astype(int)
fig, ax = plt.subplots(figsize=(5.5, 6))
im = ax.imshow(norm[np.ix_(idx, order)].T, aspect='auto', cmap='viridis')
ax.set_yticks(range(12)); ax.set_yticklabels([f'g{int(g)}' for g in order], fontsize=7)
ax.set_title('Top variable genes'); fig.colorbar(im)